# UltraGliClass stage 2 typed-decisions ABLATION (E1 / E1z / E2 / E3 / C0), one notebook

Question: did the procedural typed-decision rows added in the stage-2 pilot cost the typed-decisions `choice` accuracy (S1 0.485 -> 0.41-0.42)?
Every arm = **pilot arm C** (many-label expansion + permutation consistency, seed 42, 1,500 steps x 64 rows, init = the stage-1 model), only the data mix differs:

| arm | data change vs pilot C (all arms: 96,035 train rows, same val file `calib/stage2_pilot_val.jsonl`) | procedural-style share |
|---|---|---|
| **E1** | dedicated procedural dataset (11,750 rows) removed, filled with gliclass-v2.0 | 4.9 % (the part inside Jev; about S1's own share) |
| **E1z** | E1 and the procedural-style rows inside Jev removed too (Jev drawn from its non-procedural rest) | 0 % |
| **E2** | E1 + 3,000 rows of `LocalLLaMA/typed-decisions` **train** (customer_service + agent_trace_observability; semantic prose options) | 4.9 % |
| **E3** | pilot C + the same 3,000 semantic rows (replacing 3,000 gliclass rows) | 17.2 % |
| **C0** | pilot C unchanged (optional reproducibility / noise rerun) | 17.2 % |

E2 and E3 stay **blocked** (`SEMANTIC_DATA_APPROVED = False`) until the owner approves training on the LocalLLaMA/typed-decisions train split.
After each arm: quick eval (14 GLiClass sets + intent + typed decisions with per-type and per-workflow accuracy) -> `results/stage2_abl_<arm>_quick.json`.
**Inputs:** code dataset v4 (`ultragliclass-code`, with the ablation configs) + the S1 kernel output (as for the pilot). **Settings:** GPU T4 x2, Internet on.
Same safety features as the pilot notebook (watchdog, NCCL pre-flight, 1-GPU fallback, NaN/OOM/arm-time stops, no GPU spend beyond `TOTAL_MAX_HOURS`).


In [ ]:
# ---- knobs ----
ARMS          = ["E1", "E1z", "E2"]   # in this order: plan P1 = ["E1"]; P2 = ["E1", "E2"] (E2 needs SEMANTIC_DATA_APPROVED); see STAGE2_ABLATION_JOB_SPEC.md
INIT_MODEL    = "auto"     # S1 final_model dir, OR the S1 kernel-output root, OR "auto" = search /kaggle/input for runs/stage1_ultrabert150m/final_model
ARM_DATA      = {a: (f"configs/data_stage2_abl_{a}.yaml", f"data/stage2_abl_{a}") for a in ("E1", "E1z", "E2", "E3")}
ARM_DATA["C0"] = ("configs/data_stage2_pilot.yaml", "data/stage2_pilot")
SEM_ARMS      = ("E2", "E3")   # contain LocalLLaMA/typed-decisions TRAIN rows
SEMANTIC_DATA_APPROVED = True   # owner decision needed before E2/E3: the benchmark train split has the same 4 workflows and option strings as the eval set
PREP_ARGS     = ""                              # smoke tests: "--streaming"
CALIB_FILE    = "calib/stage1_val.jsonl"       # = data/stage1/val.jsonl used by the S1 and ModernBERT evals (same global-T fit)
CODE_INPUT    = "auto"     # auto-detect the code dataset under /kaggle/input
PREV_OUTPUT   = None       # None = fresh; "auto" = previous version's output of THIS ablation (runs/stage2_abl_*), never the S1 output or the pilot output
USE_SINGLE_GPU = False     # True = never launch 2 processes (python -u train.py on GPU 0, grad-accum doubled)
AUTO_FALLBACK_SINGLE_GPU = True   # 2-GPU run stalls/crashes (not OOM) => resume on 1 GPU in the same session
DIST_CHECK    = True       # 20-s NCCL pre-flight before the first 2-GPU launch
NCCL_P2P_DISABLE = False   # True = start with NCCL P2P off (auto-retried if the pre-flight hangs)
STALL_MINUTES = 15         # watchdog: no output for this long => kill the run
SESSION_HOURS = 12.0
RESERVE_HOURS = 0.5
ARM_MAX_HOURS = 2.0        # STOP: train budget per arm (train.py saves and exits; the notebook then stops and reports)
TOTAL_MAX_HOURS = 5.0      # STOP: no new arm starts if the session clock would pass this (GPU-hours = session hours)
EVAL_RESERVE_HOURS = 0.2   # kept free for the quick eval after an arm
QUICK_EVAL    = True
EVAL_MAX_PER_DATASET = 500
EVAL_PERMS    = 3
EVAL_CALIB_N  = 2000             # rows of CALIB_FILE used to fit the global temperature (S1: 2000)
EVAL_TYPED_MAX_CASES = None      # None = all 400 cases / 2,000 decisions (as in S1)
DROP_CKPT_AFTER_ARM = True       # delete checkpoint-* of a COMPLETED arm (final_model is kept): saves about 1.8 GB per arm
EXTRA_SET     = ["train.per_device_train_batch_size=4", "train.gradient_accumulation_steps=8", "train.gradient_checkpointing=false"]   # speed test T2: no ckpt, b4 x accum 8 x 2 GPUs = 64 (was b8 x 4 x 2 with ckpt)  # dotted overrides for every arm, e.g. ["train.max_steps=20"]
ARM_SET       = {}         # per-arm overrides, e.g. {"C": ["data.consistency.pair_prob=0.1"]}
import os, time, glob, json, sys, shutil, subprocess
T0 = time.time()
LOG_DIR = "/kaggle/working/logs"; os.makedirs(LOG_DIR, exist_ok=True)
NB_LOG = open(os.path.join(LOG_DIR, f"notebook_{time.strftime('%Y%m%d_%H%M%S')}.log"), "a", buffering=1)
def say(*a):
    line = f"[nb {time.strftime('%H:%M:%S')} +{(time.time() - T0) / 60:.1f}m] " + " ".join(str(x) for x in a)
    print(line, flush=True); NB_LOG.write(line + "\n")

def find_code_input():
    hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train.py", recursive=True)]
    hits = [h for h in hits if os.path.exists(os.path.join(h, "configs", "stage2_abl_E1.yaml"))]   # needs the v4 code (ablation configs)
    hits = [h for h in hits if not os.path.isdir(os.path.join(h, "runs"))] or hits
    hits.sort(key=lambda p: (0 if "ultragliclass-code" in p else 1, len(p)))
    say("code candidates:", hits)
    return hits[0] if hits else None

def find_prev_output():
    hits = sorted({p.split("/runs/")[0] for p in glob.glob("/kaggle/input/**/runs/stage2_abl_*/STATUS.json", recursive=True)}, key=len)
    say("previous pilot-output candidates:", hits)
    return hits[0] if hits else None

def find_init_model():
    cands = []
    if INIT_MODEL != "auto":
        cands = [INIT_MODEL] if os.path.exists(os.path.join(INIT_MODEL, "config.json")) else \
            [os.path.dirname(p) for p in glob.glob(os.path.join(INIT_MODEL, "**", "final_model", "config.json"), recursive=True)]
    else:
        cands = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/runs/stage1_ultrabert150m/final_model/config.json", recursive=True)]
    cands.sort(key=len)
    say("S1 model candidates:", cands)
    return cands[0] if cands else None

if CODE_INPUT == "auto":
    CODE_INPUT = find_code_input()
if PREV_OUTPUT == "auto":
    PREV_OUTPUT = find_prev_output()
INIT_SRC = find_init_model()
assert all(a in ARM_DATA for a in ARMS), f"unknown arm in {ARMS}"
assert SEMANTIC_DATA_APPROVED or not any(a in SEM_ARMS for a in ARMS), "E2/E3 train on LocalLLaMA/typed-decisions TRAIN rows: set SEMANTIC_DATA_APPROVED=True only with the owner's approval"
say("CODE_INPUT =", CODE_INPUT, "| INIT_SRC =", INIT_SRC, "| PREV_OUTPUT =", PREV_OUTPUT, "| ARMS =", ARMS, "| USE_SINGLE_GPU =", USE_SINGLE_GPU)
assert CODE_INPUT and os.path.exists(os.path.join(CODE_INPUT, "configs", "stage2_abl_E1.yaml")), "v4 code dataset (with configs/stage2_abl_*.yaml) not found under /kaggle/input"
assert INIT_SRC, "S1 model not found: attach the output of kernel ultragliclass-s1 (runs/stage1_ultrabert150m/final_model) or set INIT_MODEL"


In [ ]:
import shutil, subprocess
WORK = "/kaggle/working/ultragliclass"
if not os.path.exists(os.path.join(WORK, "train.py")):
    shutil.copytree(CODE_INPUT, WORK, dirs_exist_ok=True, copy_function=shutil.copyfile,
                    ignore=shutil.ignore_patterns("runs", "data", "smoke_logs", "results", "__pycache__"))
subprocess.run(f"chmod -R u+w {WORK}", shell=True)  # copytree copies read-only dir modes from /kaggle/input
os.chdir(WORK)
say("work dir", WORK)
torch_pin = subprocess.run("pip freeze | grep -E '^torch=='", shell=True, capture_output=True, text=True).stdout.strip()
open("/tmp/constraints.txt", "w").write(torch_pin + "\n")
say("torch pin:", torch_pin)
subprocess.run("pip install -q -c /tmp/constraints.txt -r requirements.txt", shell=True, check=True)
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)
import torch, transformers
NGPU = torch.cuda.device_count()
say("torch", torch.__version__, "cuda", torch.cuda.is_available(), "gpus", NGPU, "transformers", transformers.__version__)

In [ ]:
# The S1 model: validate it, then COPY it out of /kaggle/input (plain file copies, in this single process, before torchrun) so
# no rank ever reads the read-only input mount. INIT_LOCAL is what every arm gets as model.init_from.
INIT_LOCAL = "/tmp/ugc_init_model"
need = ["config.json", "tokenizer.json", "tokenizer_config.json"]
for f in need:
    assert os.path.exists(os.path.join(INIT_SRC, f)), f"S1 model dir {INIT_SRC} lacks {f}"
weights = [f for f in os.listdir(INIT_SRC) if f.endswith((".safetensors", ".bin"))]
assert weights, f"no weight file in {INIT_SRC}: {os.listdir(INIT_SRC)}"
size = sum(os.path.getsize(os.path.join(INIT_SRC, f)) for f in os.listdir(INIT_SRC))
assert size > 200e6, f"S1 model dir is only {size / 1e6:.0f} MB: incomplete"
cfg_s1 = json.load(open(os.path.join(INIT_SRC, "config.json")))
say("S1 model:", INIT_SRC, f"{size / 2**20:.0f} MB", weights, "| architectures", cfg_s1.get("architectures"), "| max_num_classes", cfg_s1.get("max_num_classes"))
shutil.rmtree(INIT_LOCAL, ignore_errors=True); os.makedirs(INIT_LOCAL)
for f in os.listdir(INIT_SRC):
    s = os.path.join(INIT_SRC, f)
    if os.path.isfile(s):
        shutil.copyfile(s, os.path.join(INIT_LOCAL, f))
        assert os.path.getsize(os.path.join(INIT_LOCAL, f)) == os.path.getsize(s), f"copy size mismatch {f}"
subprocess.run(f"chmod -R u+w {INIT_LOCAL}", shell=True)
say("S1 model copied to", INIT_LOCAL, sorted(os.listdir(INIT_LOCAL)))

In [ ]:
# Restore the previous output. Runs HERE, in the single notebook process, BEFORE torchrun: no rank ever touches
# /kaggle/input. Plain file copies (no read-only mode bits from the input mount), then chmod u+w, then verify sizes.
def restore(src_root):
    for sub in ("runs", "data", "results"):
        src = os.path.join(src_root, sub)
        if not os.path.isdir(src):
            say("restore: no", sub, "in", src_root); continue
        n = b = 0; t = time.time()
        for dirpath, _, files in os.walk(src):
            rel = os.path.relpath(dirpath, src_root)
            os.makedirs(rel, exist_ok=True)
            for f in files:
                s, d = os.path.join(dirpath, f), os.path.join(rel, f)
                size = os.path.getsize(s)
                if not (os.path.exists(d) and os.path.getsize(d) == size):
                    shutil.copyfile(s, d)
                assert os.path.getsize(d) == size, f"size mismatch {d}"
                n += 1; b += size
                if size > 50e6: say(f"restore: copied {d} ({size / 2**20:.0f} MB)")
        say(f"restore: {sub}: {n} files, {b / 2**30:.2f} GB in {time.time() - t:.0f}s")
    subprocess.run("chmod -R u+w runs data results 2>/dev/null; true", shell=True)

if PREV_OUTPUT:
    say("restore from", PREV_OUTPUT)
    restore(PREV_OUTPUT)
for st in glob.glob("runs/*/STATUS.json"):
    s = json.load(open(st)); say("found", st, s.get("state"), "step", s.get("global_step"), "last", s.get("last_checkpoint"))
for c in sorted(glob.glob("runs/*/checkpoint-*")):
    say("checkpoint", c, sorted(os.listdir(c)))

In [ ]:
# Per-arm data: each arm has its own recipe and data dir; prepared lazily, right before the arm trains.
sys.path.insert(0, WORK)
from ugc.config import load_config as _lc
assert os.path.exists("calib/stage2_pilot_val.jsonl"), "calib/stage2_pilot_val.jsonl missing from the code dataset (v4)"
os.makedirs("results", exist_ok=True)
PROC_SHARE = {}
def prep_arm_data(arm):
    recipe, ddir = ARM_DATA[arm]
    if arm in SEM_ARMS:
        assert SEMANTIC_DATA_APPROVED, "E2/E3 are blocked until SEMANTIC_DATA_APPROVED = True"
    if not os.path.exists(os.path.join(ddir, "train.jsonl")):
        say("preparing data for arm", arm, ":", recipe, PREP_ARGS)
        subprocess.run(f"python -u prepare_data.py --recipe {recipe} {PREP_ARGS}", shell=True, check=True)
    else:
        say("data already present for arm", arm, ":", ddir)
    man = json.load(open(os.path.join(ddir, "manifest.json")))
    shutil.copyfile(os.path.join(ddir, "manifest.json"), f"results/stage2_abl_{arm}_data_manifest.json")
    _t = _lc(f"configs/stage2_abl_{arm}.yaml", [f"model.init_from={INIT_LOCAL}"] + EXTRA_SET)["train"]
    need_rows = int(_t["max_steps"]) * int(_t["per_device_train_batch_size"]) * int(_t.get("gradient_accumulation_steps", 1)) * \
        (NGPU if NGPU >= 2 and not USE_SINGLE_GPU else 1)      # 1500 x 64 = 96,000 on the real run
    used = {k: v for k, v in man["sources"].items() if v.get("train_rows", 0) > 0}
    bad = [k for k in used if any(x in k.lower() for x in ("dbpedia", "bitext", "banking77", "clinc", "massive", "snips", "phrasebank", "sst5"))]
    sem = {k: v["train_rows"] for k, v in used.items() if "localllama/typed-decisions" in k.lower()}
    proc_dedicated = sum(v["train_rows"] for k, v in used.items() if k.startswith("tasksource/procedural-typed-decisions"))
    n_proc = n_tot = 0                                         # procedural-style rows incl. the ones inside Jev (source tag)
    with open(os.path.join(ddir, "train.jsonl")) as f:
        for line in f:
            n_tot += 1; n_proc += ("procedural-typed-decisions" in json.loads(line)["source"])
    PROC_SHARE[arm] = round(n_proc / max(n_tot, 1), 4)
    say(f"[{arm}] train rows {man['train_rows']} val {man['val_rows']} | procedural-style rows {n_proc} ({100 * PROC_SHARE[arm]:.1f} %) | semantic rows {sem} | sources", {k: v["train_rows"] for k, v in used.items()})
    assert not bad, f"excluded / eval-set source in the data: {bad}"
    assert man["train_rows"] >= need_rows, f"only {man['train_rows']} train rows, need {need_rows}"
    assert bool(sem) == (arm in SEM_ARMS), f"[{arm}] semantic (LocalLLaMA typed-decisions train) rows present={bool(sem)} but expected {arm in SEM_ARMS}"
    if arm in SEM_ARMS:
        assert sum(sem.values()) == 3000, sem
    exp = {"E1": (0, 0.03, 0.07), "E1z": (0, 0.0, 0.0), "E2": (0, 0.03, 0.07), "E3": (11750, 0.15, 0.20), "C0": (11750, 0.15, 0.20)}[arm]
    assert (proc_dedicated == exp[0] and exp[1] <= PROC_SHARE[arm] <= exp[2]), \
        f"[{arm}] procedural share {PROC_SHARE[arm]} / dedicated {proc_dedicated} not as designed {exp}"
    return man

In [ ]:
# Launch with live, unbuffered streaming to this notebook AND to /kaggle/working/logs/, a no-output watchdog, a NCCL pre-flight,
# a single-GPU fallback, then the quick eval, one arm after the other.
import signal, threading, queue, re
sys.path.insert(0, WORK)
from ugc.config import load_config
LAST_OOM = False

def run_streamed(cmd, env_extra, log_name, stall_min):
    global LAST_OOM
    LAST_OOM = False
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TORCH_NCCL_ASYNC_ERROR_HANDLING": "1",
           "TORCH_NCCL_DUMP_ON_TIMEOUT": "1", "NCCL_DEBUG": "WARN", **env_extra}
    path = os.path.join(LOG_DIR, f"{log_name}_{time.strftime('%H%M%S')}.log")
    say("RUN", cmd, "| env", env_extra, "| log", path)
    p = subprocess.Popen(cmd, shell=True, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, start_new_session=True)
    q = queue.Queue()
    threading.Thread(target=lambda: ([q.put(l) for l in p.stdout], q.put(None)), daemon=True).start()
    last, stalled = time.time(), False
    with open(path, "a", buffering=1) as f:
        while True:
            try:
                line = q.get(timeout=10)
            except queue.Empty:
                if time.time() - last > stall_min * 60:
                    say(f"WATCHDOG: no output for {stall_min} min -> killing process group"); stalled = True
                    os.killpg(p.pid, signal.SIGTERM); time.sleep(30)
                    if p.poll() is None: os.killpg(p.pid, signal.SIGKILL)
                    break
                continue
            if line is None: break
            if re.search(r"OutOfMemoryError|CUDA out of memory", line): LAST_OOM = True
            last = time.time(); print(line, end="", flush=True); f.write(line)
    rc = p.wait()
    say("EXIT", rc, "stalled" if stalled else "", "OOM" if LAST_OOM else "")
    return rc, stalled

def elapsed_h():
    return (time.time() - T0) / 3600

def arm_budget_h():
    return min(ARM_MAX_HOURS, TOTAL_MAX_HOURS - elapsed_h() - EVAL_RESERVE_HOURS, SESSION_HOURS - elapsed_h() - RESERVE_HOURS)

def arm_sets(arm):
    return [f"model.init_from={INIT_LOCAL}"] + EXTRA_SET + ARM_SET.get(arm, [])

def train_cmd(arm, single):
    sets = [f"train.time_budget_hours={max(0.1, arm_budget_h()):.2f}"] + arm_sets(arm)
    accum = int(load_config(f"configs/stage2_abl_{arm}.yaml", arm_sets(arm))["train"].get("gradient_accumulation_steps", 1))
    if single and NGPU > 1:
        sets.append(f"train.gradient_accumulation_steps={accum * NGPU}")
    launcher = "python -u" if single else f"torchrun --standalone --nproc_per_node {NGPU}"
    return f"{launcher} train.py --config configs/stage2_abl_{arm}.yaml --resume auto --set {' '.join(sets)}"

single = USE_SINGLE_GPU or NGPU < 2
nccl_env = {"NCCL_P2P_DISABLE": "1"} if NCCL_P2P_DISABLE else {}
if not single and DIST_CHECK:
    rc, st = run_streamed(f"torchrun --standalone --nproc_per_node {NGPU} scripts/dist_check.py", nccl_env, "dist_check", 3)
    if rc != 0 and not nccl_env:
        say("pre-flight failed: retrying with NCCL_P2P_DISABLE=1"); nccl_env = {"NCCL_P2P_DISABLE": "1"}
        rc, st = run_streamed(f"torchrun --standalone --nproc_per_node {NGPU} scripts/dist_check.py", nccl_env, "dist_check_p2poff", 3)
    if rc != 0:
        say("pre-flight failed twice: using 1 GPU"); single = True

def read_status(out_dir):
    p = os.path.join(out_dir, "STATUS.json")
    return json.load(open(p)) if os.path.exists(p) else {}

def run_arm(arm):
    global single
    cfg = load_config(f"configs/stage2_abl_{arm}.yaml", arm_sets(arm))
    out_dir = cfg["train"]["output_dir"]
    res_path = f"results/stage2_abl_{arm}_quick.json"
    rec = {"arm": arm, "out_dir": out_dir, "state": None, "t0_hours": round(elapsed_h(), 3)}
    st = read_status(out_dir)
    if st.get("state") == "complete" and os.path.isdir(st.get("final_model", "")):
        say(f"ARM {arm}: already complete (restored from PREV_OUTPUT): skipping training")
    else:
        prep_arm_data(arm)
        if arm_budget_h() < 0.4:
            rec["state"] = "not_started_budget"; say(f"ARM {arm}: NOT STARTED, only {arm_budget_h():.2f} h of budget left"); return rec
        say(f"===== ARM {arm}: {cfg['train']['max_steps']} steps, budget {arm_budget_h():.2f} h, many_label={cfg['data']['many_label']['enabled']}, "
            f"consistency={cfg['data']['consistency']['enabled']} =====")
        rc, stalled = run_streamed(train_cmd(arm, single), ({"CUDA_VISIBLE_DEVICES": "0"} if single else nccl_env),
                                   f"train_{arm}_{'1gpu' if single else 'ddp'}", STALL_MINUTES)
        st = read_status(out_dir)
        if (rc != 0 or stalled) and not LAST_OOM and not single and AUTO_FALLBACK_SINGLE_GPU and st.get("state") != "complete":
            say(f"ARM {arm}: 2-GPU run failed/stalled -> resuming on 1 GPU from the newest checkpoint")
            single = True
            rc, stalled = run_streamed(train_cmd(arm, True), {"CUDA_VISIBLE_DEVICES": "0"}, f"train_{arm}_1gpu_fallback", STALL_MINUTES)
            st = read_status(out_dir)
        rec["oom"] = bool(LAST_OOM)
    hb = os.path.join(out_dir, "HEARTBEAT.json")
    if os.path.exists(hb): say("last heartbeat:", open(hb).read()[:300])
    rec["state"] = st.get("state", "no_status")
    rec["status"] = st
    print(json.dumps(st, indent=2))
    if rec["state"] != "complete":
        say(f"ARM {arm}: state={rec['state']} -> STOP (needs_resume = used the {ARM_MAX_HOURS} h arm budget; nan_stop = NaN loss; running = crash/OOM/stall)")
        return rec
    if QUICK_EVAL and not os.path.exists(res_path):
        cmd = (f"python -u eval.py --model {st['final_model']} --suites gliclass intent typed --max_per_dataset {EVAL_MAX_PER_DATASET} "
               f"--permutations {EVAL_PERMS} --calib_file {CALIB_FILE} --calib_n {EVAL_CALIB_N} --out {res_path}" +
               (f" --typed_max_cases {EVAL_TYPED_MAX_CASES}" if EVAL_TYPED_MAX_CASES else ""))
        rec["eval_rc"], _ = run_streamed(cmd, {}, f"quick_eval_{arm}", 30)
    if os.path.exists(res_path):
        r = json.load(open(res_path)); d = r["datasets"]
        rec["eval"] = {"avg_macro_f1": r.get("gliclass_avg_macro_f1"), "global_T": r.get("global_T"),
                       "typed_acc": (r.get("typed_decisions") or {}).get("T1", {}).get("accuracy"),
                       "typed_by_type": (r.get("typed_decisions") or {}).get("T1", {}).get("accuracy_by_type"),
                       "typed_by_workflow": (r.get("typed_decisions") or {}).get("T1", {}).get("accuracy_by_workflow"),
                       "procedural_share": PROC_SHARE.get(arm),
                       "flip_many_label": {k: d[k]["perm_flip_rate"] for k in ("banking77", "clinc150", "massive", "SetFit/20_newsgroups") if k in d}}
    if DROP_CKPT_AFTER_ARM:
        for c in glob.glob(os.path.join(out_dir, "checkpoint-*")):
            shutil.rmtree(c, ignore_errors=True)
    rec["wall_hours"] = round(elapsed_h() - rec["t0_hours"], 3)
    return rec

SUMMARY = []
def write_summary():
    json.dump({"session_hours": round(elapsed_h(), 3), "arms": SUMMARY, "init_model_src": INIT_SRC, "procedural_share": PROC_SHARE}, open("results/stage2_abl_summary.json", "w"), indent=2)

for arm in ARMS:
    if elapsed_h() > TOTAL_MAX_HOURS - 0.4:
        say(f"STOP: session clock {elapsed_h():.2f} h is past the {TOTAL_MAX_HOURS} h total cap; arm {arm} not started"); break
    rec = run_arm(arm); SUMMARY.append(rec); write_summary()
    say(f"ARM {arm} done: state={rec['state']} wall={rec.get('wall_hours')} h eval={rec.get('eval')}")
    if rec["state"] != "complete":
        break

In [ ]:
print(json.dumps({"session_hours": round(elapsed_h(), 2), "arms": [{k: v for k, v in r.items() if k != "status"} for r in SUMMARY]}, indent=2))
for r in SUMMARY:
    s = r.get("status", {})
    say(f"ARM {r['arm']}: {r['state']} step {s.get('global_step')}/{s.get('max_steps')} train_loss {s.get('train_loss')} "
        f"final_eval_loss {(s.get('final_eval') or {}).get('eval_loss')} peak_mem {(s.get('final_eval') or {}).get('peak_mem_gb')} avg_F1 {(r.get('eval') or {}).get('avg_macro_f1')}")
if any(r["state"] != "complete" for r in SUMMARY) or len(SUMMARY) < len(ARMS):
    say("NOT ALL ARMS COMPLETE: save this version; to continue attach its output, set PREV_OUTPUT='auto' and ARMS to the remaining arms (+ the unfinished one).")

In [ ]:
for out in sorted(glob.glob("runs/stage2_abl_*")):          # unfinished arms keep their newest checkpoint only
    ck = sorted(glob.glob(os.path.join(out, "checkpoint-*")), key=lambda p: int(p.rsplit("-", 1)[1]))
    for c in ck[:-1]:
        shutil.rmtree(c)
subprocess.run("du -sh runs data results /kaggle/working/logs 2>/dev/null", shell=True)
say("session hours used:", round(elapsed_h(), 2)); NB_LOG.close()